# Notebook 16 — Prompt Engineering Practicum
**AI Models and Agents for Business · Week 8 · Session 16**

---

### The five prompt patterns

| # | Pattern | Best for |
|---|---|---|
| 1 | Zero-shot | Quick tasks with clear intent |
| 2 | Few-shot | Matching a format or style you define |
| 3 | Chain-of-thought (structured steps) | Complex reasoning, multi-step problems you want to check |
| 4 | Role + constraints | High-stakes outputs needing guardrails |
| 5 | Structured output (JSON) | Downstream processing, databases, APIs |

All five patterns run against the **same business scenario** so you can compare outputs directly.

**API used:** Groq (free). See the final cell to swap to Claude or GPT-4o.

---

## Cell 1 — Setup

In [ ]:
!pip install groq --quiet
from groq import Groq
import json, textwrap

# Your Groq key comes from Colab Secrets (key icon on the left, name GROQ_API_KEY, Notebook access on).
# If it is not there, you will be asked to paste it in a box that hides what you type.
def _get_key():
    try:
        from google.colab import userdata
        return userdata.get("GROQ_API_KEY")
    except Exception:
        import getpass
        return getpass.getpass("Paste your Groq API key (starts with gsk_): ")

GROQ_API_KEY = _get_key()
client = Groq(api_key=GROQ_API_KEY)

# ✏️ CHANGE ME — try "openai/gpt-oss-120b" for better quality (slower; uses more of the free daily limit)
MODEL = "openai/gpt-oss-20b"

# Groq retired llama-3.1-8b-instant for free accounts in August 2026, so this notebook uses
# openai/gpt-oss-20b. It is a reasoning model: before it answers, it writes hidden "thinking"
# tokens. They count toward max_tokens and are billed as output tokens. The lines below keep
# that thinking short and add room for it, so every max_tokens in this notebook still means
# "room for the answer".
REASONING_EFFORT = "low"      # "low", "medium" or "high"
REASONING_HEADROOM = 300      # extra tokens reserved for the hidden thinking
_groq_create = client.chat.completions.create
def _create_with_reasoning(*args, **kwargs):
    if "gpt-oss" in str(kwargs.get("model", "")):
        kwargs.setdefault("reasoning_effort", REASONING_EFFORT)
        kwargs.setdefault("include_reasoning", False)
        if kwargs.get("max_tokens"):
            kwargs["max_tokens"] += REASONING_HEADROOM
    return _groq_create(*args, **kwargs)
client.chat.completions.create = _create_with_reasoning

def call(prompt, system=None, max_tokens=400, temperature=0.3):
    msgs = []
    if system:
        msgs.append({"role": "system", "content": system})
    msgs.append({"role": "user", "content": prompt})
    r = client.chat.completions.create(
        model=MODEL, messages=msgs,
        max_tokens=max_tokens, temperature=temperature
    )
    return r.choices[0].message.content or ""

def show(label, text):
    print(f"\n{'━'*60}")
    print(f"  {label}")
    print(f"{'━'*60}")
    print(textwrap.fill(text, width=70, subsequent_indent='  '))

print(f"✅ Ready. Model: {MODEL}")
print("Scenario: Risk summary for an AI chatbot at a regional bank.")

## The scenario
Your company is deploying an AI chatbot for first-line customer complaints at a regional bank. You need a risk summary for senior leadership.

## Cell 2 — Pattern 1: Zero-shot

In [ ]:
# ✏️ CHANGE ME — modify the task or context
ZERO_SHOT = """Write a risk summary for deploying an AI chatbot to handle
first-line customer complaints at a regional bank."""

result_zs = call(ZERO_SHOT)
show("Pattern 1: Zero-shot", result_zs)

## Cell 3 — Pattern 2: Few-shot

In [ ]:
# ✏️ CHANGE ME — try different example formats (table, numbered list)
FEW_SHOT = """Write a risk summary in this format. Examples from other deployments:

EXAMPLE 1 — AI resume screener:
DEPLOYMENT: AI resume screener for HR department
RISKS:
  [HIGH]   Demographic bias may violate EEOC guidelines
  [MEDIUM] Candidates cannot appeal automated rejections
  [LOW]    Requires retraining when job requirements change
RECOMMENDATION: Pilot with human review of all rejections first.

EXAMPLE 2 — AI loan underwriting:
DEPLOYMENT: AI loan underwriting assistant
RISKS:
  [HIGH]   Fair lending compliance risk if model uses race proxies
  [HIGH]   Explainability required for adverse action notices
  [MEDIUM] Model drift as economic conditions change
RECOMMENDATION: Human sign-off on all loans above $50K.

Now write the same format for:
DEPLOYMENT: AI chatbot for first-line customer complaints at a regional bank
"""

result_fs = call(FEW_SHOT)
show("Pattern 2: Few-shot", result_fs)

### 💬 Zero-shot vs. few-shot
- Which output is more useful to a non-technical executive?
- What did the examples force the model to include that it missed in zero-shot?

## Cell 4 — Pattern 3: Chain-of-thought (structured steps)

The classic version of this pattern was simply adding "think step by step." Reasoning models like the `gpt-oss` model used here already think privately before they answer (Session 14), so that phrase adds less than it used to. The lasting value of this pattern is the **visible structure**: you name the steps, and the answer comes back in pieces you can check one at a time.

Treat those written steps as a summary you can audit, not as a guaranteed record of how the model actually reached its answer.

In [ ]:
# ✏️ CHANGE ME — try removing "Think through this step by step" and compare
COT = """A regional bank is deploying an AI chatbot for first-line customer complaint
handling (password resets, dispute inquiries, account questions). The chatbot
will escalate complex cases to human agents.

Think through this step by step:
1. What are the most likely failure modes?
2. Which failures would cause the most harm to customers?
3. Which would create regulatory or legal risk?
4. What mitigation would you recommend for the top two risks?

After your reasoning, provide a final 3-sentence executive recommendation.
"""

result_cot = call(COT, max_tokens=600)
show("Pattern 3: Chain-of-thought", result_cot)

## Cell 5 — Pattern 4: Role + constraints

In [ ]:
# ✏️ CHANGE ME — try: bank examiner, plaintiffs attorney, CTO
ROLE_SYS = """You are a senior bank examiner with 20 years at the OCC (Office of
the Comptroller of the Currency). You review AI deployments for regulatory compliance.

Rules:
- Never describe a risk as 'low' without citing a specific mitigation in place
- Always cite the specific regulation at risk (UDAAP, ECOA, FCRA, etc.)
- Flag any area where you lack sufficient information to assess risk
- Be direct; do not soften findings"""

ROLE_PROMPT = """Review this AI deployment and provide your examination findings:

The bank plans to deploy an AI chatbot for first-line customer complaints including
password resets, dispute inquiries, and account questions. The system uses a large
language model from a third-party vendor. Human escalation is triggered by the
chatbot itself."""

result_role = call(ROLE_PROMPT, system=ROLE_SYS, max_tokens=500)
show("Pattern 4: Role + constraints", result_role)

### 💬 Discuss
- Which pattern gave the most actionable output for a compliance use case?
- The role prompt asked the model to flag gaps in information. Did it?
- What's the risk of assigning a role the model doesn't actually have expertise in?

## Cell 6 — Pattern 5: Structured output (JSON)

In [ ]:
STRUCT_SYS = """You are a risk assessment API. Respond ONLY with valid JSON and nothing else.
No preamble, no explanation, no markdown code fences. Only the raw JSON object."""

# ✏️ CHANGE ME — add more fields to the schema
STRUCT_PROMPT = """Assess risks of deploying an AI chatbot for bank customer complaints.
Return a JSON object with this exact schema:
{
  "deployment": "string",
  "overall_risk_level": "LOW | MEDIUM | HIGH | CRITICAL",
  "risks": [
    {
      "risk_id": "R1",
      "category": "string",
      "severity": "LOW | MEDIUM | HIGH",
      "description": "string — one sentence",
      "mitigation": "string — one sentence"
    }
  ],
  "recommended_action": "string — one sentence",
  "review_date": "string — ISO date 6 months from today"
}
Include exactly 4 risks. Return only the JSON."""

raw = call(STRUCT_PROMPT, system=STRUCT_SYS, temperature=0.1)

try:
    clean = raw.strip().strip('`').replace('json\n','').strip()
    parsed = json.loads(clean)
    print("✅ Valid JSON returned\n")
    print(f"Deployment:   {parsed['deployment']}")
    print(f"Overall risk: {parsed['overall_risk_level']}")
    print(f"Action:       {parsed['recommended_action']}")
    print()
    print(f"{'ID':<5} {'Category':<16} {'Severity':<10} Description")
    print("-"*70)
    for r in parsed['risks']:
        desc = r['description'][:45] + '...' if len(r['description']) > 45 else r['description']
        print(f"{r['risk_id']:<5} {r['category']:<16} {r['severity']:<10} {desc}")
except json.JSONDecodeError:
    print("⚠️  Model did not return valid JSON. Raw output:")
    print(raw)
    print()
    print("Tip: open-source models sometimes add markdown fences or preamble.")
    print("Try MODEL = 'openai/gpt-oss-120b' for better instruction-following.")

### 💬 Structured output discussion
- If this JSON went into a risk database, what would that enable?
- What happens if the model returns valid JSON only 90% of the time?
- Why is `temperature=0.1` especially important here?
- Note: smaller open-source models sometimes fail JSON format. This is a real production concern — try the 70B model if you see failures.

## Cell 7 — Hallucination lab

In [ ]:
hallucination_prompts = [
    "What was the exact regulatory fine the OCC issued to Wells Fargo in Q3 2024 related to AI?",
    "Cite the specific section of the EU AI Act governing bank chatbots and the exact penalty amount.",
    "What is Klarna's current employee headcount as of last month?",
]

for i, prompt in enumerate(hallucination_prompts, 1):
    result = call(prompt, max_tokens=150, temperature=0.1)
    print(f"\nQuestion {i}: {prompt}")
    print(f"Response: {result[:300]}")

### 💬 Discussion
- How confidently did the model answer? Did it acknowledge uncertainty?
- If a financial analyst used these in a board presentation, what could go wrong?
- Add this to a prompt: *"If you are not certain, say so rather than guessing."* Does it help?

## Cell 8 — Apply to your own project

In [ ]:
# ✏️ CHANGE ME — pick the best pattern for your project domain
# Pattern: _______________

MY_SYSTEM = None  # or add a system prompt for role + constraints

MY_PROMPT = """
Replace this with a prompt relevant to your team project.
Pick the pattern that fits best and note why you chose it.
"""

my_result = call(MY_PROMPT, system=MY_SYSTEM, max_tokens=500)
show("Your prompt", my_result)

---
## 🔄 Model Swap: Anthropic Claude or OpenAI GPT-4o

The patterns above work identically on any model. Here's how to switch.

In [ ]:
# MODEL SWAP — Anthropic Claude
# Uncomment if you have an Anthropic key

# !pip install anthropic --quiet
# import anthropic

# ANTHROPIC_KEY = "sk-ant-PASTE_YOUR_KEY_HERE"
# claude = anthropic.Anthropic(api_key=ANTHROPIC_KEY)

# # Same few-shot prompt, Claude client
# result_claude = claude.messages.create(
#     model="claude-3-5-haiku-20241022",
#     max_tokens=400,
#     temperature=0.3,
#     messages=[{"role": "user", "content": FEW_SHOT}]
# )
# print("── Claude (few-shot) ──")
# print(result_claude.content[0].text)

# ---

# MODEL SWAP — OpenAI GPT-4o
# Uncomment if you have an OpenAI key

# !pip install openai --quiet
# from openai import OpenAI

# OPENAI_KEY = "sk-PASTE_YOUR_KEY_HERE"
# oai = OpenAI(api_key=OPENAI_KEY)

# result_gpt = oai.chat.completions.create(
#     model="gpt-4o-mini",
#     messages=[{"role": "user", "content": FEW_SHOT}],
#     max_tokens=400,
# )
# print("── GPT-4o mini (few-shot) ──")
# print(result_gpt.choices[0].message.content)

# Note: OpenAI uses the same .choices[0].message.content format as Groq — 
# they intentionally made their APIs compatible.

print("Uncomment one of the blocks above to compare models.")
print()
print("Key API differences:")
print("  Groq / OpenAI: response.choices[0].message.content")
print("  Anthropic:     response.content[0].text")
print()
print("Interesting experiment: run the JSON structured output prompt on all three models.")
print("Which one follows the schema most reliably?")

## ✅ Notebook 16 complete — Reflection questions

These prep you for **Quiz 5** (Session 20).

1. What is the difference between a **system prompt** and a **user prompt**?
2. When would you choose **chain-of-thought** over **zero-shot**? Give a specific business example.
3. What is **hallucination** and why is it especially dangerous in financial services?
4. Why does JSON structured output require a very low temperature?
5. The JSON cell sometimes fails on smaller models but works on larger ones. What does that tell you about the relationship between model size and instruction-following?

---
*Next session: RAG Architecture — how to ground a model's answers in your own documents.*